# Validação de Dados - Views BigQuery

Este notebook valida se as views criadas no BigQuery estão retornando os dados corretamente, especialmente focando na resolução do problema de duplicação nas médias.

In [ ]:
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "directed-mender-489100-m3"
client = bigquery.Client(project=PROJECT_ID)

def run_query(sql):
    return client.query(sql).to_dataframe()

## 1. Comparação de Contagem de Linhas

A `vw_analise_games` (detalhada) deve ter mais linhas que a `vw_analise_games_unica` (agrupada) se existirem jogos multiplataforma ou multigênero.

In [ ]:
sql_counts = """
SELECT 
    'Detalhada (vw_analise_games)' as view_name, count(*) as total_rows
FROM `directed-mender-489100-m3.games_analytics.vw_analise_games` 
UNION ALL
SELECT 
    'Unica (vw_analise_games_unica)' as view_name, count(*) as total_rows
FROM `directed-mender-489100-m3.games_analytics.vw_analise_games_unica` 
"""
df_counts = run_query(sql_counts)
df_counts

## 2. Teste de Duplicação (Caso Real)

Vamos pegar um jogo conhecido por ser multiplataforma e ver como ele aparece em cada view.

In [ ]:
nome_jogo = "The Witcher 3: Wild Hunt" # Exemplo común

print(f"--- Analisando: {nome_jogo} ---")

sql_detalhe = f"""
SELECT nome_jogo, plataforma, genero 
FROM `directed-mender-489100-m3.games_analytics.vw_analise_games` 
WHERE nome_jogo LIKE '%{nome_jogo}%'
"""

sql_unica = f"""
SELECT nome_jogo, plataformas, generos 
FROM `directed-mender-489100-m3.games_analytics.vw_analise_games_unica` 
WHERE nome_jogo LIKE '%{nome_jogo}%'
"""

print("Linhas na View Detalhada:")
display(run_query(sql_detalhe))

print("\nLinhas na View Unificada:")
display(run_query(sql_unica))

## 3. Verificação de KPIs

Verificando se a média de nota crítica (Metascore) não possui valores nulos gritantes ou escalas erradas.

In [ ]:
sql_kpis = """
SELECT 
    AVG(nota_critica) as media_metascore,
    AVG(nota_usuarios) as media_usuarios,
    MAX(nota_critica) as max_metascore,
    MIN(nota_critica) as min_metascore
FROM `directed-mender-489100-m3.games_analytics.vw_analise_games_unica` 
WHERE nota_critica IS NOT NULL
"""
run_query(sql_kpis)